# Train the custom CNN on Colab

This notebook runs the project’s `SimpleCNN` and augmentation setup on the full [Melanoma Cancer Image Dataset](https://www.kaggle.com/datasets/bhaveshmittal/melanoma-cancer-dataset). It makes a fixed 80/20 validation split from `train/` (seed 42), selects the best validation checkpoint, and evaluates `test/` once at the end.

Choose **Runtime → Change runtime type → T4 GPU** before running. Colab sessions can disconnect; the checkpoint and results are saved in `/content/skin-lesion-cnn-results/` and can be downloaded from the Files pane. This is an academic classifier, not a clinical diagnostic tool.


In [ ]:
%pip -q install kagglehub
import kagglehub
from pathlib import Path

root = Path(kagglehub.dataset_download("bhaveshmittal/melanoma-cancer-dataset"))
print("Downloaded to:", root)

def find_dataset_root(root):
    candidates = [root] + [p for p in root.rglob("train") if p.is_dir()]
    for candidate in candidates:
        base = candidate if candidate.name != "train" else candidate.parent
        if all((base / split / label).is_dir() for split in ("train", "test") for label in ("Benign", "Malignant")):
            return base
    raise FileNotFoundError(f"Could not find train/test class folders under {root}")

data_dir = find_dataset_root(root)
for split in ("train", "test"):
    print(split, {label: len(list((data_dir / split / label).glob("*.jpg"))) for label in ("Benign", "Malignant")})


In [ ]:
import json, random, time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

SEED = 42
EPOCHS = 20
BATCH_SIZE = 32
OUT = Path("/content/skin-lesion-cnn-results")
OUT.mkdir(exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

class MelonaDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.transform = transform
        self.classes = sorted(p.name for p in Path(data_dir).iterdir() if p.is_dir())
        self.samples = [(str(path), label) for label, name in enumerate(self.classes)
                        for path in sorted((Path(data_dir) / name).glob("*.jpg"))]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
        return image, label

train_folder = data_dir / "train"
base = MelonaDataset(train_folder)
train_indices, val_indices = [], []
for label in range(len(base.classes)):
    indices = [i for i, (_, y) in enumerate(base.samples) if y == label]
    random.Random(SEED + label).shuffle(indices)
    n_val = round(len(indices) * 0.20)
    val_indices.extend(indices[:n_val])
    train_indices.extend(indices[n_val:])
train_indices.sort(); val_indices.sort()
assert not set(train_indices).intersection(val_indices)
print("Classes:", base.classes)
print("Train:", len(train_indices), "Val:", len(val_indices))
print("Train counts:", dict(Counter(base.samples[i][1] for i in train_indices)))
print("Val counts:", dict(Counter(base.samples[i][1] for i in val_indices)))

# Match train.py: average the channel mean and std of each training batch.
stats_dataset = Subset(MelonaDataset(train_folder, transforms.ToTensor()), train_indices)
stats_loader = DataLoader(stats_dataset, batch_size=64, shuffle=False, num_workers=2)
mean, std, batches = torch.zeros(3), torch.zeros(3), 0
for images, _ in stats_loader:
    mean += images.mean(dim=[0, 2, 3])
    std += images.std(dim=[0, 2, 3])
    batches += 1
mean = (mean / batches).tolist()
std = (std / batches).tolist()
print("Mean:", mean, "Std:", std)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])
val_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
train_dataset = Subset(MelonaDataset(train_folder, train_transform), train_indices)
val_dataset = Subset(MelonaDataset(train_folder, val_transform), val_indices)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=device.type == "cuda")
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=device.type == "cuda")


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classificateur = nn.Sequential(nn.Flatten(), nn.Linear(64 * 28 * 28, 256), nn.ReLU(),
                                           nn.Dropout(p=0.5), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classificateur(self.bloc3(self.bloc2(self.bloc1(x))))

def run_epoch(model, loader, loss_fn, optimizer=None):
    model.train(optimizer is not None)
    loss_total = correct = count = 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        if optimizer is not None:
            optimizer.zero_grad()
        with torch.set_grad_enabled(optimizer is not None):
            logits = model(images)
            loss = loss_fn(logits, labels)
            if optimizer is not None:
                loss.backward()
                optimizer.step()
        count += len(labels)
        loss_total += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
    return loss_total / count, correct / count

model = SimpleCNN(len(base.classes)).to(device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = {key: [] for key in ("train_loss", "train_acc", "val_loss", "val_acc")}
best_val_acc, best_val_loss, best_epoch = -1.0, float("inf"), 0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn)
    for key, value in (("train_loss", train_loss), ("train_acc", train_acc),
                       ("val_loss", val_loss), ("val_acc", val_acc)):
        history[key].append(value)
    if val_acc > best_val_acc or (val_acc == best_val_acc and val_loss < best_val_loss):
        best_val_acc, best_val_loss, best_epoch = val_acc, val_loss, epoch
        torch.save({"model_type": "simple_cnn", "model_state_dict": model.state_dict(),
                    "num_classes": len(base.classes), "classes": base.classes, "mean": mean, "std": std,
                    "best_val_acc": best_val_acc, "best_val_loss": best_val_loss, "best_epoch": best_epoch,
                    "split_seed": SEED, "val_fraction": 0.20,
                    "train_images": len(train_dataset), "val_images": len(val_dataset)}, OUT / "checkpoint_aug.pth")
    (OUT / "history.json").write_text(json.dumps(history, indent=2))
    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {train_loss:.4f}, acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f}, acc {val_acc:.4f} | {time.time()-start:.1f}s", flush=True)
print(f"Best validation epoch: {best_epoch}, accuracy: {best_val_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ("loss", "acc")):
    ax.plot(history[f"train_{metric}"], label="train")
    ax.plot(history[f"val_{metric}"], label="validation")
    ax.set(xlabel="epoch (0-based)", ylabel=metric); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "training_curves.png", dpi=150); plt.show()


In [ ]:
# Run only after all training epochs. Do not use test results to choose a checkpoint.
checkpoint = torch.load(OUT / "checkpoint_aug.pth", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
test_dataset = MelonaDataset(data_dir / "test", val_transform)
assert test_dataset.classes == base.classes
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=device.type == "cuda")
confusion = torch.zeros((len(base.classes), len(base.classes)), dtype=torch.int64)
test_loss_total = 0.0
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device, non_blocking=True))
        test_loss_total += loss_fn(logits, labels.to(device, non_blocking=True)).item() * len(labels)
        preds = logits.argmax(1).cpu()
        for actual, predicted in zip(labels, preds):
            confusion[actual, predicted] += 1

correct = confusion.diag().sum().item()
result = {"model": "SimpleCNN with augmentation", "classes": base.classes,
          "train_images": len(train_dataset), "val_images": len(val_dataset), "test_images": len(test_dataset),
          "split_seed": SEED, "best_epoch": checkpoint["best_epoch"],
          "best_val_accuracy": checkpoint["best_val_acc"],
          "test_accuracy": correct / len(test_dataset), "test_correct": correct,
          "test_loss": test_loss_total / len(test_dataset),
          "confusion_matrix": confusion.tolist(),
          "note": "Rows are actual labels; columns are predicted labels. Test evaluated once after checkpoint selection."}
(OUT / "test_result.json").write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(confusion, cmap="Blues")
for row in range(len(base.classes)):
    for col in range(len(base.classes)):
        ax.text(col, row, int(confusion[row, col]), ha="center", va="center")
ax.set_xticks(range(len(base.classes)), base.classes)
ax.set_yticks(range(len(base.classes)), base.classes)
ax.set(xlabel="Predicted", ylabel="Actual", title="Custom CNN test confusion matrix")
fig.tight_layout(); fig.savefig(OUT / "test_confusion_matrix.png", dpi=150); plt.show()


## Keep the results

In Colab’s **Files** pane, open `/content/skin-lesion-cnn-results/` and download `test_result.json`, `history.json`, the plots, and `checkpoint_aug.pth`. The checkpoint is large; download it if you want to rerun evaluation without retraining.
